In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import sys
import re
import os

root_dir = "/projects/bbhh/suyufeng/enzyme_specificity"

sys.path.append(f"{root_dir}/src")
sys.path.append(f"{root_dir}/src/other_softwares/grover_software")

# Download enzyme active info from brenda
See Dataset.utils (download_uniprot_file)

# Create the reaction features
See Dataset.utils (get_reaction_feature)

# Create the enzyme features
See Dataset.utils (get_enzyme_feature)

# Create positive samples dataset

In [2]:
df = pd.read_csv(f"{root_dir}/data/brenda/data_cofactor.csv", sep=',').dropna(subset=['left', 'right', 'uniprot', 'ecnumber'])
enzyme_df = pd.read_csv(f"{root_dir}/data/brenda/enzymes.csv", sep=',')
reaction_df = pd.read_csv(f"{root_dir}/data/brenda/reaction.csv", sep=',')

uniprot_dict = {uniprot: index for index, uniprot in enumerate(enzyme_df['uniprots'].values.tolist())}
reaction_dict = {reaction: index for index, reaction in enumerate(reaction_df['reactions'].values.tolist())}

data = {
    'reaction': [],
    'enzyme': [],
    'ecnumber': []
}

for left, right, uniprot, ecnumber in zip(df['left'], df['right'], df['uniprot'], df['ecnumber']):
    if uniprot in uniprot_dict and left + '>>' + right in reaction_dict:
        data['reaction'].append(reaction_dict[left + '>>' + right])
        data['enzyme'].append(uniprot_dict[uniprot])
        data['ecnumber'].append(ecnumber)

data = pd.DataFrame(data).sample(frac=1).reset_index()
data.to_csv(f"{root_dir}/data/positive_data.csv", index=False)


# Calculate max_length of reaction

In [3]:
import lmdb
import pickle

reaction_save_lmdb_path = f"{root_dir}/data/brenda/reaction_features.lmdb"
db = lmdb.open(
    reaction_save_lmdb_path,
    map_size=10*(1024*1024*1024),   # 10GB
    create=False,
    subdir=False,
    readonly=True,
    lock=False,
    readahead=False,
    meminit=False,
)
with db.begin() as txn:
    keys = list(txn.cursor().iternext(values=False))

max_n_atoms = 0

for key in keys:
     with db.begin(write=False, buffers=True) as txn:
        # key = str(key).encode()
        value = txn.get(key)
        if value is None:
            raise KeyError
        data = pickle.loads(value)
        max_n_atoms = max(max_n_atoms, data['element'].shape[0])
        # break
print(max_n_atoms)

135


# Create grover embedding

In [4]:
# 1. Tou Tou is going to create smile only csv
import pandas as pd
import os

df = pd.read_csv(f"{root_dir}/data/brenda/reaction.csv", sep=',')
results = [smile for smile in df['substrates']]
data = {
    "substrates": results
}
data = pd.DataFrame(data)
data.to_csv(f"{root_dir}/data/brenda/reaction_smiles.csv", index=False)

In [5]:
# 2. Get npz feature
print(f"python {root_dir}/src/other_softwares/grover_software/scripts/save_features.py --data_path {root_dir}/data/brenda/reaction_smiles.csv  \
                                --save_path {root_dir}/data/brenda/reaction.npz   \
                                --features_generator fgtasklabel \
                                --restart")

python /projects/bbhh/suyufeng/enzyme_specificity/src/other_softwares/grover_software/scripts/save_features.py --data_path /projects/bbhh/suyufeng/enzyme_specificity/data/brenda/reaction_smiles.csv                                  --save_path /projects/bbhh/suyufeng/enzyme_specificity/data/brenda/reaction.npz                                   --features_generator fgtasklabel                                 --restart


In [13]:
# 3. Get build vocab
print(f"python {root_dir}/src/other_softwares/grover_software/scripts/build_vocab.py --data_path {root_dir}/data/brenda/reaction_smiles.csv \
                             --vocab_save_folder {root_dir}/data/brenda/grover_vocab  \
                             --dataset_name brenda")

                            

python /projects/bbhh/suyufeng/enzyme_specificity/src/other_softwares/grover_software/scripts/build_vocab.py --data_path /projects/bbhh/suyufeng/enzyme_specificity/data/brenda/reaction_smiles.csv                              --vocab_save_folder /projects/bbhh/suyufeng/enzyme_specificity/data/brenda/grover_vocab                               --dataset_name brenda


In [14]:
# 4. Get fingerprint
print(f"python main.py fingerprint --data_path {root_dir}/data/brenda/reaction_smiles.csv --features_path {root_dir}/data/brenda/reaction.npz --checkpoint_path {root_dir}/data/pretrain_model/grover_large.pt --fingerprint_source both --output {root_dir}/data/brenda/fingerprint.npz --save_lmdb_path {root_dir}/data/brenda/grover_fingerprint.lmdb --fingerprint_source both")

python main.py fingerprint --data_path /projects/bbhh/suyufeng/enzyme_specificity/data/brenda/reaction_smiles.csv --features_path /projects/bbhh/suyufeng/enzyme_specificity/data/brenda/reaction.npz --checkpoint_path /projects/bbhh/suyufeng/enzyme_specificity/data/pretrain_model/grover_large.pt --fingerprint_source both --output /projects/bbhh/suyufeng/enzyme_specificity/data/brenda/fingerprint.npz --save_lmdb_path /projects/bbhh/suyufeng/enzyme_specificity/data/brenda/grover_fingerprint.lmdb --fingerprint_source both


# Create morgan embedding

In [6]:
from rdkit.Chem import AllChem
from rdkit import Chem

path = f"{root_dir}/data/brenda/reaction_smiles.csv"
df = pd.read_csv(path, sep=',')
results = []
for smile in df['substrates']:
    m1 = Chem.MolFromSmiles(smile)
    result = np.array(AllChem.GetMorganFingerprintAsBitVect(m1,2,nBits=1024))
    results.append(result)
np.save(f"{root_dir}/data/brenda/morgan_fingerprint.npy", np.array(results))

[01:34:51] WARNING: not removing hydrogen atom without neighbors
[01:34:51] WARNING: not removing hydrogen atom without neighbors
[01:34:51] WARNING: not removing hydrogen atom without neighbors
[01:34:52] WARNING: not removing hydrogen atom without neighbors
[01:34:53] WARNING: not removing hydrogen atom without neighbors
[01:34:53] WARNING: not removing hydrogen atom without neighbors
[01:34:53] WARNING: not removing hydrogen atom without neighbors
[01:34:53] WARNING: not removing hydrogen atom without neighbors
[01:34:53] WARNING: not removing hydrogen atom without neighbors
[01:35:01] WARNING: not removing hydrogen atom without neighbors
[01:35:01] WARNING: not removing hydrogen atom without neighbors
[01:35:01] WARNING: not removing hydrogen atom without neighbors
[01:35:01] WARNING: not removing hydrogen atom without neighbors
[01:35:01] WARNING: not removing hydrogen atom without neighbors
[01:35:01] WARNING: not removing hydrogen atom without neighbors
[01:35:12] WARNING: not r

# Generate negative samples

In [3]:
from Datasets.utils import generate_negative_sample

pdf = pd.read_csv(f"{root_dir}/data/positive_data.csv", sep=',', index_col=False)
pdf['difficulty'] = [-1] * pdf['ecnumber'].values.shape[0]
pdf['fake_ecnumber'] = pdf['ecnumber']
pdf['label'] = [1] * pdf['ecnumber'].values.shape[0]
dfs = [pdf]
for same_digits in range(0, 6):
    df = generate_negative_sample(pdf, same_digits=same_digits, num_negative_enzyme=1, has_positive_sample=False)
    df['difficulty'] = [same_digits] * df.index.shape[0]
    dfs.append(df)
df = pd.concat(dfs)
df.to_csv(f"{root_dir}/data/data.csv", index=False)

/u/suyufeng/.conda/envs/revae/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Start generating negative samples!


100%|██████████| 187195/187195 [00:03<00:00, 53222.42it/s]


Start generating negative samples!


100%|██████████| 187195/187195 [00:03<00:00, 54281.39it/s]


...

100%|██████████| 187195/187195 [00:02<00:00, 73363.72it/s]


Start generating negative samples!


100%|██████████| 187195/187195 [00:03<00:00, 61505.72it/s]


Start generating negative samples!


100%|██████████| 187195/187195 [00:02<00:00, 78105.59it/s]


# MISC: Add more information for docking simulation

In [4]:
import pandas as pd

t = pd.read_csv(f"{root_dir}/data/data.csv", sep=',')
enzyme_df = pd.read_csv(f"{root_dir}/data/brenda/enzymes.csv", sep=',')
reaction_df = pd.read_csv(f"{root_dir}/data/brenda/reaction.csv", sep=',')

enzmye_dict = {index: uniprot for index, uniprot in enumerate(enzyme_df['uniprots'])}
reaction_dict = {index: (reaction, substrate) for index, (reaction, substrate) in enumerate(zip(reaction_df['reactions'], reaction_df['substrates']))}

uniprots = []
reactions = []
substrates = []
activesites = []

df = pd.read_csv(f"{root_dir}/data/brenda/data_cofactor.csv", sep=',')
uniprot_active_site_dict = {uniprot: active_site for uniprot, active_site in zip(df['uniprot'], df['activesites'])}

for enzyme, reaction in zip(t['enzyme'], t['reaction']):
    uniprot = enzmye_dict[enzyme]
    reaction, substrate = reaction_dict[reaction]
    uniprots.append(uniprot)
    reactions.append(reaction)
    substrates.append(substrate)
    activesites.append(uniprot_active_site_dict[uniprot])

t['uniprot'] = uniprots
t['reaction'] = reactions
t['substrate'] = substrates
t['active_site'] = activesites

t.to_csv(f"{root_dir}/data/to_docking.csv", index=False)

# MISC: create cofactor names

In [19]:
import pandas as pd
from tqdm import tqdm
import glob

root_dir = "/projects/bbhh/suyufeng/enzyme_specificity"

data = {
    "name": [],
    "inchi": []
}
name_dict = {}
for name in tqdm(glob.glob(f"{root_dir}/data/brenda/raw_data/ligand/*.csv")):
    # if '7.2.4.5.csv' not in name:
    #     continue
    try:
        t = pd.read_csv(name, sep='\t', header=None)
        for name, type, inchi in zip(t[0], t[2], t[4]):
            if 'Cofactor' in type:
                if name not in name_dict:
                    name_dict[name] = inchi
                    data['name'].append(name)
                    data['inchi'].append(inchi)
    except:
        pass
# print(data)
data = pd.DataFrame(data)
data.to_csv("to_ocean.csv", index=False)

100%|██████████| 8281/8281 [00:26<00:00, 311.25it/s]


# MISC: post-process docking data and training data

In [3]:
import pandas as pd

root_dir = "/projects/bbto/suyufeng/enzyme_specificity"

df = pd.read_csv(f"{root_dir}/data/brenda/data.csv", sep=',')
enzyme_df = pd.read_csv(f"{root_dir}/data/brenda/enzymes.csv", sep=',')
reaction_df = pd.read_csv(f"{root_dir}/data/brenda/reaction.csv", sep=',')

docking_index_df = pd.read_csv(f"{root_dir}/data/brenda/docking_subset_data.csv", sep=',')

docking_index_dict = {(substrate, uniprot):index for index, (substrate, uniprot) in enumerate(zip(docking_index_df['substrate'], docking_index_df['uniprot']))}

enzyme_dict = {uniprot: index for index, uniprot in enumerate(enzyme_df['uniprots'])}
reaction_dict = {reaction: index for index, reaction in enumerate(reaction_df['reactions'])}

substrate_enzyme_dict = {}

# index,reaction,enzyme,ecnumber,difficulty,fake_ecnumber,label,uniprot,substrate,active_site

data = {
    "enzyme": [],
    "reaction": [],
    "label": [],
    "ecnumber": [],
    "difficulty": [],
    "fake_ecnumber": [],
    "structure_index": [],
    "substrate": []
}

cnt = 0
for enzyme, reaction, label, ecnumber, difficulty, fake_ecnumber, substrate, uniprot in zip(df['enzyme'], df['reaction'], df['label'], df['ecnumber'], df['difficulty'], df['fake_ecnumber'], df['substrate'], df['uniprot']):
    if (uniprot, substrate) in substrate_enzyme_dict:
        continue
    
    substrate_enzyme_dict[(uniprot, substrate)] = 1
    if (substrate, uniprot) in docking_index_dict:
        data['structure_index'].append(docking_index_dict[(substrate, uniprot)])
        cnt += 1
    else:
        data['structure_index'].append(-1)
    
    data['enzyme'].append(enzyme)
    data['reaction'].append(reaction_dict[reaction])
    data['label'].append(label)
    data['ecnumber'].append(ecnumber)
    data['difficulty'].append(difficulty)
    data['fake_ecnumber'].append(fake_ecnumber)
    data['substrate'].append(substrate)
    
print(cnt)
data = pd.DataFrame(data)
data.to_csv(f"{root_dir}/data/brenda/final_data/data.csv", index=False)


258061


In [1]:
import pandas as pd
import os
from tqdm import tqdm

root_dir = "/projects/bbto/suyufeng/enzyme_specificity"
df = pd.read_csv(f"{root_dir}/data/brenda/final_data/data.csv", sep=',')
# df = df.loc[df["structure_index"] != -1, :].reset_index(drop=True)

def save_data_csv(datas, file_name, substrate_constraint=None, enzyme_constraint=None):
    data_df = {
        "enzyme": [],
        "reaction": [],
        "label": [],
        "ecnumber": [],
        "difficulty": [],
        "fake_ecnumber": [],
        "structure_index": []
    }
    if substrate_constraint is not None:
        substrate_dict = {substrate: 1 for substrate in substrate_constraint}
    else:
        substrate_dict = None
    if enzyme_constraint is not None:
        enzyme_dict = {enzyme: 1 for enzyme in enzyme_constraint}
    else:
        enzyme_dict = None

    for (enzyme, reaction, label, ecnumber, difficulty, fake_ecnumber, structure_index, substrate) in tqdm(datas):
        if substrate_dict is not None and substrate not in substrate_dict:
            continue
        if enzyme_dict is not None and enzyme not in enzyme_dict:
            continue
        data_df['enzyme'].append(enzyme)
        data_df['reaction'].append(reaction)
        data_df['label'].append(label)
        data_df['ecnumber'].append(ecnumber)
        data_df['difficulty'].append(difficulty)
        data_df['fake_ecnumber'].append(fake_ecnumber)
        data_df['structure_index'].append(structure_index)

    data_df = pd.DataFrame(data_df)
    data_df.to_csv(f"{root_dir}/data/brenda/final_data/{file_name}.csv", sep=',', index=False)

In [2]:
datas = [
    (enzyme, reaction, label, ecnumber, difficulty, fake_ecnumber, structure_index, substrate)
    for enzyme, reaction, label, ecnumber, difficulty, fake_ecnumber, structure_index, substrate in zip(df['enzyme'], df['reaction'], df['label'], df['ecnumber'], df['difficulty'], df['fake_ecnumber'], df['structure_index'], df['substrate'])
]
substrates = list(set([substrate for substrate in df['substrate']]))
enzymes = list(set([enzyme for enzyme in df['enzyme']]))
# random split
import random
random.shuffle(datas)
random.shuffle(substrates)
random.shuffle(enzymes)

os.makedirs(f"{root_dir}/data/brenda/final_data/random_split", exist_ok=True)
for i in range(4):
    training_datas = datas[:int(len(datas) * 0.25 * i)] + datas[int(len(datas) * 0.25 * (i + 1)):]
    
    testing_datas = datas[int(len(datas) * 0.25 * i):int(len(datas) * 0.25 * (i + 1))]
    val_datas = testing_datas[:int(len(testing_datas) * 0.3)]
    testing_datas = testing_datas[int(len(testing_datas) * 0.3):]

    save_data_csv(training_datas, f"random_split/training_datas_{i}")
    save_data_csv(val_datas, f"random_split/val_datas_{i}")
    save_data_csv(testing_datas, f"random_split/testing_datas_{i}")
    
os.makedirs(f"{root_dir}/data/brenda/final_data/reaction_split", exist_ok=True)
os.makedirs(f"{root_dir}/data/brenda/final_data/enzyme_split", exist_ok=True)
os.makedirs(f"{root_dir}/data/brenda/final_data/all_split", exist_ok=True)
for i in range(4):
    # enzyme
    training_enzymes = enzymes[:int(len(enzymes) * 0.25 * i)] + enzymes[int(len(enzymes) * 0.25 * (i + 1)):]

    testing_enzymes = enzymes[int(len(enzymes) * 0.25 * i):int(len(enzymes) * 0.25 * (i + 1))]
    val_enzymes = testing_enzymes[:int(len(testing_enzymes) * 0.3)]
    testing_enzymes = testing_enzymes[int(len(testing_enzymes) * 0.3):]
    if len(val_enzymes) == 0:
        val_enzymes = testing_enzymes[:2]
        testing_enzymes = testing_enzymes[2:]

    # substrate
    training_substrates = substrates[:int(len(substrates) * 0.25 * i)] + substrates[int(len(substrates) * 0.25 * (i + 1)):]
    
    testing_substrates = substrates[int(len(substrates) * 0.25 * i):int(len(substrates) * 0.25 * (i + 1))]
    val_substrates = testing_substrates[:int(len(testing_substrates) * 0.3)]
    testing_substrates = testing_substrates[int(len(testing_substrates) * 0.3):]
    if len(val_substrates) == 0:
        val_substrates = testing_substrates[:2]
        testing_substrates = testing_substrates[2:]

    save_data_csv(datas, f"enzyme_split/training_datas_{i}", enzyme_constraint=training_enzymes)
    save_data_csv(datas, f"enzyme_split/val_datas_{i}", enzyme_constraint=val_enzymes)
    save_data_csv(datas, f"enzyme_split/testing_datas_{i}", enzyme_constraint=testing_enzymes)

    save_data_csv(datas, f"reaction_split/training_datas_{i}", substrate_constraint=training_substrates)
    save_data_csv(datas, f"reaction_split/val_datas_{i}",substrate_constraint=val_substrates)
    save_data_csv(datas, f"reaction_split/testing_datas_{i}", substrate_constraint=testing_substrates)

    save_data_csv(datas, f"all_split/training_datas_{i}", substrate_constraint=training_substrates, enzyme_constraint=training_enzymes)
    save_data_csv(datas, f"all_split/val_datas_{i}",substrate_constraint=val_substrates, enzyme_constraint=val_enzymes)
    save_data_csv(datas, f"all_split/testing_datas_{i}", substrate_constraint=testing_substrates,enzyme_constraint=testing_enzymes)

save_data_csv(datas, "big_datas")

100%|██████████| 586515/586515 [00:00<00:00, 1292973.08it/s]


# Generate spreadsheet for baselines

In [1]:
import pandas as pd
import numpy as np
def amend_meta(enzyme_path, reaction_path, data_path, tag):
    enzyme_df = pd.read_csv(enzyme_path, sep=',')
    reaction_df = pd.read_csv(reaction_path, sep=',')
    data_df = pd.read_csv(data_path, sep=',')

    enzyme_sequences = []
    substrate_smiles = []

    enzyme_dict = {index: sequence for index, sequence in enumerate(enzyme_df['sequences'])}
    reaction_dict = {index: substrate for index, substrate in enumerate(reaction_df['substrates'])}

    for enzyme, reaction in zip(data_df['enzyme'], data_df['reaction']):
        enzyme_sequences.append(enzyme_dict[enzyme])
        substrate_smiles.append(reaction_dict[reaction])

    data_df['enzyme_sequence'] = enzyme_sequences
    data_df['substrate_smile'] = substrate_smiles

    data_df.to_csv(f"/projects/bbto/suyufeng/enzyme_specificity/saved_data/{tag}.csv", index=False)
    # for index, df in enumerate(np.array_split(data_df, 7)):
    #     df.to_csv(f"/projects/bbto/suyufeng/enzyme_specificity/saved_data/{tag}_{index}.csv", index=False)

# for tag in  ["Duf", "Gt_acceptor", "halogenase", "Nitrilase", "Phosphatase", "Thiolase", "Esterase", "experiment"]:
for tag in ["2023_brenda"]:
    root_dir = f"/projects/bbto/suyufeng/enzyme_specificity/data/small_family/{tag}"
    enzyme_path = f"{root_dir}/enzymes.csv"
    reaction_path = f"{root_dir}/reactions.csv"
    data_path = f"{root_dir}/big_datas.csv"
    amend_meta(enzyme_path, reaction_path, data_path, tag)

# enzyme_path = "/projects/bbto/suyufeng/enzyme_specificity/data/brenda/enzymes.csv"
# reaction_path = "/projects/bbto/suyufeng/enzyme_specificity/data/brenda/reaction.csv"
# data_path = "/projects/bbto/suyufeng/enzyme_specificity/data/brenda/final_data/big_datas.csv"
# amend_meta(enzyme_path, reaction_path, data_path, "brenda")

    